# Track B: fast handoff files from frozen Granite (no GPU needed)

Track A's LightGBM model is ready to combine with the text score, but the fine-tuned handoff files (step 4) take hours on a laptop GPU. This notebook builds a **complete, consistent set of the same four files in a few minutes**, using the frozen Granite embeddings already cached by step 2 and the logistic regression from step 2.

| File | Text model trained on | Scores | Track A uses it to |
|---|---|---|---|
| `text_oof_train.csv` | 5 folds of 2018-2024 (labels by Dec 31, 2024) | 2024 rows, out-of-fold | train |
| `text_val_2025.csv` | all of 2018-2024 | 2025 validation rows | validate (with vs. without text) |
| `text_oof_final.csv` | 5 folds of 2018-2025 (labels by Dec 31, 2025) | 2024-2025 rows, out-of-fold | retrain for the final ranking |
| `text_2026.csv` | all of 2018-2025 | all 72,670 scoreboard rows | score the final ranking |

**Why every file comes from the same recipe:** Track A's model learns what a given text score *means* from the training file, then applies that to 2026. If the files came from different text models, that lesson wouldn't transfer.

**The tradeoff:** frozen text has an AUC of 0.80 on 2025, against 0.854 for the fine-tuned model. When the fine-tuned set is ready (step 4), it uses the same file names and format, so swapping is one line.

Files go to `data/trackb/handoff_frozen/`.

## 1. Load the texts, labels and cached embeddings

In [1]:
import time, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA = next(p for p in [Path("../data"), Path("data")] if p.exists()).resolve()
OUT = DATA / "trackb"
HANDOFF = OUT / "handoff_frozen"
HANDOFF.mkdir(exist_ok=True)

texts = pd.read_parquet(OUT / "texts.parquet")
def aligned(split, emb_name):
    df = texts[texts.split == split].sort_values("cve_id", ignore_index=True)
    ids = pd.read_parquet(OUT / f"emb_{emb_name}_ids.parquet")["cve_id"]
    assert ids.equals(df["cve_id"]), f"{split}: embedding order doesn't match"
    return df, np.load(OUT / f"emb_{emb_name}.npy")

train, E_train = aligned("train", "train")      # 2018-2024
val, E_val = aligned("val", "val")              # 2025 validation
s26, E_26 = aligned("score26", "score26")       # 2026 scoreboard

def labeled(df, deadline):
    return (df["kev_date"].notna() & (df["kev_date"] <= pd.Timestamp(deadline))).astype(int).to_numpy()

y24 = labeled(train, "2024-12-31")
both = pd.concat([train, val], ignore_index=True)
E_both = np.vstack([E_train, E_val])
y25 = labeled(both, "2025-12-31")
print(f"2018-2024: {len(train):,} rows, {y24.sum()} positives | 2018-2025: {len(both):,} rows, {y25.sum()} positives")

2018-2024: 165,885 rows, 684 positives | 2018-2025: 201,166 rows, 803 positives


## 2. The text model (same as step 2) and out-of-fold scoring

The same pipeline as step 2: standardize the 384 embedding numbers, then a logistic regression with `class_weight="balanced"` and fixed settings (`C=1.0`).

**Out-of-fold:** split the training rows into 5 stratified parts, train on 4, and score the 5th. Every row Track A trains on is scored by a model that never saw its label.

In [2]:
def text_model():
    return make_pipeline(StandardScaler(), LogisticRegression(C=1.0, class_weight="balanced", max_iter=3000))

def out_of_fold(E, y, score_mask):
    scores = np.full(len(y), np.nan)
    for fit_idx, out_idx in StratifiedKFold(5, shuffle=True, random_state=0).split(E, y):
        m = text_model().fit(E[fit_idx], y[fit_idx])
        target = out_idx[score_mask[out_idx]]
        scores[target] = m.predict_proba(E[target])[:, 1]
    return scores[score_mask]

t0 = time.time()
mask24 = (train["year"] == 2024).to_numpy()
oof_train = pd.DataFrame({"cve_id": train.loc[mask24, "cve_id"], "text_score": out_of_fold(E_train, y24, mask24)})

val_score = text_model().fit(E_train, y24).predict_proba(E_val)[:, 1]
text_val = pd.DataFrame({"cve_id": val["cve_id"], "text_score": val_score})

mask_final = both["year"].isin([2024, 2025]).to_numpy()
oof_final = pd.DataFrame({"cve_id": both.loc[mask_final, "cve_id"], "text_score": out_of_fold(E_both, y25, mask_final)})

text_2026 = pd.DataFrame({"cve_id": s26["cve_id"],
                          "text_score": text_model().fit(E_both, y25).predict_proba(E_26)[:, 1]})
print(f"all four files computed in {(time.time() - t0) / 60:.1f} min")

all four files computed in 1.6 min


## 3. Sanity checks

- **Validation:** the 2025 score should reproduce step 2's frozen result (4 hits@243, AUC 0.80).
- **Out-of-fold:** on Track A's own 2024 training set (published January-September 2024, 33 positives, top 198), the out-of-fold text score should be informative, but not suspiciously perfect.

In [3]:
def hits_at_k(y, s, k):
    tie = np.random.default_rng(0).permutation(len(s))
    return int(y[np.lexsort([tie, -s])[:k]].sum())

yv = val["y"].to_numpy()
print(f"val 2025: hits@243 {hits_at_k(yv, val_score, 243)}, AUC {roc_auc_score(yv, val_score):.3f}")

ta = pd.read_parquet(DATA / "backtest/train_2024.parquet").sort_values("cve_id", ignore_index=True)
ta = ta.merge(oof_train, on="cve_id", how="left")
print(f"Track A 2024 rows with a text score: {ta.text_score.notna().mean():.1%}")
print(f"OOF on Track A 2024 set: hits@198 {hits_at_k(ta.y.to_numpy(), ta.text_score.fillna(0).to_numpy(), 198)}, "
      f"AUC {roc_auc_score(ta.y, ta.text_score.fillna(0)):.3f}")
for name, df, n_expected in [("text_oof_train", oof_train, mask24.sum()), ("text_val_2025", text_val, len(val)),
                             ("text_oof_final", oof_final, mask_final.sum()), ("text_2026", text_2026, 72_670)]:
    assert len(df) == n_expected and df.cve_id.is_unique and df.text_score.notna().all(), name
print("row counts, unique IDs and no blanks: OK")

val 2025: hits@243 4, AUC 0.800


Track A 2024 rows with a text score: 100.0%
OOF on Track A 2024 set: hits@198 2, AUC 0.837
row counts, unique IDs and no blanks: OK


## 4. Save

In [4]:
for name, df in [("text_oof_train.csv", oof_train), ("text_val_2025.csv", text_val),
                 ("text_oof_final.csv", oof_final), ("text_2026.csv", text_2026)]:
    df.to_csv(HANDOFF / name, index=False)
    print(f"{name:20} {len(df):>7,} rows")

text_oof_train.csv    39,868 rows
text_val_2025.csv     35,281 rows


text_oof_final.csv    75,149 rows


text_2026.csv         72,670 rows


## How Track A plugs it in

```python
text = pd.read_csv("data/trackb/handoff_frozen/text_oof_train.csv")   # cve_id, text_score
train = train.merge(text, on="cve_id", how="left")
# Each file comes from different model fits, so use the percentile rank, not the raw score:
train["text_pct"] = train.groupby("snap")["text_score"].rank(pct=True)
# add "text_pct" to the LightGBM features
```

1. **Train** on 2024 with `text_oof_train.csv`.
2. **Validate** on 2025 with `text_val_2025.csv`: LightGBM with vs. without `text_pct`, using the same `hits_at_k` and paired bootstrap.
3. **Final:** retrain on 2024 + 2025 with `text_oof_final.csv`, and score 2026 with `text_2026.csv`.
4. **Later:** swap the folder `handoff_frozen/` for `handoff/` (fine-tuned) and re-run. The file names and columns are identical.